# 05c — GAT Baselines · Phase 5C

**Purpose:** Load trained vanilla-GAT and GATv2 baselines, evaluate on the geographic test split, and build the **architecture ablation ladder** against the full GAT (Phase 5A) and all Phase 4 baselines.

**Run AFTER:**
```
python scripts/phase5c_train_gat_baselines.py --arch GAT_vanilla
python scripts/phase5c_train_gat_baselines.py --arch GATv2
```

**Why these two models:**
- **Vanilla GAT** — same 2-layer / 4-head / hidden-256 backbone as the full GAT, but a single-output regression head trained with MSE. No Gaussian NLL head, no MC Dropout, no calibration. This *isolates* what the uncertainty machinery contributes: full GAT vs vanilla GAT changes exactly one thing — the head.
- **GATv2** — same backbone with GATv2Conv (dynamic attention, Brody et al. 2022). Pre-empts the reviewer question "why GAT and not GATv2?".

**Critical rule:** NEVER re-apply QuantileTransformer at eval — `y` is already transformed. Only `inverse_transform` before metrics.

In [1]:
import os, sys, json, pickle
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed
from wildfire_gnn.models.gnn import build_model, count_parameters
from wildfire_gnn.evaluation.metrics import (
    r2_score, mae_score, spearman_rho, brier_score,
    expected_calibration_error, binned_metrics
)

config = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

p            = config['paths']
GRAPH_PATH   = PROJECT_ROOT / p['graph_data']
TRANS_PATH   = PROJECT_ROOT / p['target_transformer']
CKPT_DIR     = PROJECT_ROOT / 'checkpoints'
TBL_DIR      = PROJECT_ROOT / 'reports' / 'tables'
FIG_DIR      = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load ALL prior results for the comparison ladder:
#   Phase 4 tabular + CNN baselines, AND the Phase 5A full-GAT/GCN/SAGE table.
BASELINES = {}
for csv in ['phase4_baseline_metrics.csv', 'phase4b_cnn_metrics.csv',
            'phase5a_all_models_comparison.csv']:
    path = TBL_DIR / csv
    if path.exists():
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            BASELINES[row['model']] = row.to_dict()

print(f'Project root    : {PROJECT_ROOT}')
print(f'Baselines loaded: {list(BASELINES.keys())}')

Project root    : d:\wildfire\spatiotemporal_wildfire_gnn
Baselines loaded: ['Naive Mean', 'Ridge Regression', 'Random Forest', 'XGBoost', '2D CNN (spatial)', 'GAT', 'GCN', 'GraphSAGE']


In [4]:
graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print('Graph loaded:')
print(f'  num_nodes     : {graph.num_nodes:,}')
print(f'  num_features  : {graph.num_node_features}')
print(f'  train         : {int(graph.train_mask.sum()):,}')
print(f'  val           : {int(graph.val_mask.sum()):,}')
print(f'  test          : {int(graph.test_mask.sum()):,}')
print(f'  y mean        : {float(graph.y.mean()):.4f}  (should be near 0)')
print(f'  y std         : {float(graph.y.std()):.4f}   (should be near 1)')

# Assertions — identical protocol to Phase 5A
assert graph.num_node_features == 61, f'Expected 61, got {graph.num_node_features}'
assert abs(float(graph.y.mean())) < 0.5, 'y not transformed or double-transformed!'
assert (graph.train_mask & graph.val_mask).sum() == 0, 'Train/Val overlap!'
assert (graph.train_mask & graph.test_mask).sum() == 0, 'Train/Test overlap!'
assert graph.val_mask.sum() > 0, 'val_mask is zero!'
print('\n✓ All graph assertions passed')

Graph loaded:
  num_nodes     : 327,405
  num_features  : 61
  train         : 237,304
  val           : 32,570
  test          : 57,531
  y mean        : 0.0077  (should be near 0)
  y std         : 0.9930   (should be near 1)

✓ All graph assertions passed


In [5]:
# Evaluation helper — vanilla baselines are POINT predictors (MSE-trained,
# regression head). No MC Dropout needed: a single deterministic forward pass
# in eval() mode. We still inverse-transform BEFORE any metric.

def evaluate_baseline(arch):
    ckpt_path = CKPT_DIR / f'gnn_{arch.lower()}_best.pt'
    if not ckpt_path.exists():
        print(f'  Checkpoint not found: {ckpt_path.name}')
        print(f'  Run: python scripts/phase5c_train_gat_baselines.py --arch {arch}')
        return None

    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = build_model(
        architecture = arch,
        in_channels  = config['model']['in_channels'],
        hidden       = config['model']['hidden_channels'],
        num_layers   = config['model'].get('num_layers', 2),
        heads        = config['model'].get('heads', 4),
        dropout      = config['model'].get('dropout', 0.3),
    )
    model.load_state_dict(ckpt['model_state'])
    model.eval()   # deterministic — NO dropout for a point-prediction baseline

    with torch.no_grad():
        mean, _ = model(graph.x, graph.edge_index)
        mean_pred = mean[graph.test_mask].numpy()

    with open(TRANS_PATH, 'rb') as f:
        transformer = pickle.load(f)
    y_pred_bp = transformer.inverse_transform(mean_pred.reshape(-1, 1)).ravel()
    y_true_bp = graph.y_raw[graph.test_mask].numpy().ravel()

    m = dict(
        model    = model.name,
        r2       = r2_score(y_true_bp, y_pred_bp),
        mae      = mae_score(y_true_bp, y_pred_bp),
        spearman = spearman_rho(y_true_bp, y_pred_bp),
        brier    = brier_score(y_true_bp, y_pred_bp),
        ece      = expected_calibration_error(y_true_bp, y_pred_bp),
        n_test   = len(y_true_bp),
        params   = count_parameters(model),
        y_true_bp = y_true_bp, y_pred_bp = y_pred_bp,
    )
    print(f'  {model.name:16s} R²={m["r2"]:.4f}  MAE={m["mae"]:.5f}  '
          f'Spearman={m["spearman"]:.4f}  ECE={m["ece"]:.5f}  params={m["params"]:,}')
    return m

print('Evaluating GAT baselines (point predictors, eval mode):\n')
res_vanilla = evaluate_baseline('GAT_vanilla')
res_gatv2   = evaluate_baseline('GATv2')

Evaluating GAT baselines (point predictors, eval mode):

  GAT_vanilla      R²=0.6543  MAE=0.01393  Spearman=0.8996  ECE=0.01026  params=150,273
  Checkpoint not found: gnn_gatv2_best.pt
  Run: python scripts/phase5c_train_gat_baselines.py --arch GATv2


In [6]:
# Persist per-model metric rows in the SAME format as Phase 5A
rows = []
for r in [res_vanilla, res_gatv2]:
    if r is None:
        continue
    row = {k: r[k] for k in ['model','r2','mae','spearman','brier','ece','n_test']}
    rows.append(row)
    out = TBL_DIR / f"phase5c_{r['model'].lower()}_metrics.csv"
    pd.DataFrame([row]).to_csv(out, index=False)
    print(f'  Saved {out.name}')

phase5c_df = pd.DataFrame(rows)
print()
print(phase5c_df.to_string(index=False))

  Saved phase5c_gat_vanilla_metrics.csv

      model       r2      mae  spearman    brier      ece  n_test
GAT_vanilla 0.654337 0.013927  0.899634 0.000482 0.010261   57531


In [ ]:
# ── ARCHITECTURE ABLATION LADDER ──
# Full GAT (Phase 5A) vs vanilla GAT vs GATv2. The full-GAT row is read from
# the Phase 5A comparison table so the numbers are exactly the published ones.
full_gat = BASELINES.get('GAT', None)

ladder = []
if full_gat is not None:
    ladder.append({'variant': 'GAT (full: NLL + MC Dropout + calib.)',
                   'conv': 'GATConv', 'head': 'Gaussian NLL', 'loss': 'gaussian_nll',
                   'r2': full_gat.get('r2'), 'mae': full_gat.get('mae'),
                   'spearman': full_gat.get('spearman'), 'ece': full_gat.get('ece')})
if res_vanilla is not None:
    ladder.append({'variant': 'GAT (vanilla: single head)',
                   'conv': 'GATConv', 'head': 'Regression', 'loss': 'MSE',
                   'r2': res_vanilla['r2'], 'mae': res_vanilla['mae'],
                   'spearman': res_vanilla['spearman'], 'ece': res_vanilla['ece']})
if res_gatv2 is not None:
    ladder.append({'variant': 'GATv2 (dynamic attention)',
                   'conv': 'GATv2Conv', 'head': 'Regression', 'loss': 'MSE',
                   'r2': res_gatv2['r2'], 'mae': res_gatv2['mae'],
                   'spearman': res_gatv2['spearman'], 'ece': res_gatv2['ece']})

ladder_df = pd.DataFrame(ladder)
print('  ARCHITECTURE ABLATION LADDER (test split, original BP scale)')
print('  ' + '='*78)
print(ladder_df.to_string(index=False))

ladder_path = TBL_DIR / 'phase5c_ablation_ladder.csv'
ladder_df.to_csv(ladder_path, index=False)
print(f'\n  Saved: {ladder_path.name}')

# Interpretation guard — quantify what the uncertainty head buys
if full_gat is not None and res_vanilla is not None:
    d_r2 = full_gat.get('r2', 0) - res_vanilla['r2']
    print(f"\n  Full GAT vs vanilla GAT:  ΔR² = {d_r2:+.4f}")
    if d_r2 >= 0:
        print('  → The Gaussian NLL head + MC Dropout do not cost predictive accuracy')
        print('    (and add calibrated uncertainty the vanilla model cannot provide).')
    else:
        print('  → The NLL head trades a little raw R² for calibrated uncertainty —')
        print('    an honest, expected trade-off worth stating explicitly in the paper.')

In [ ]:
# Full comparison: every prior model + the two new GAT baselines, ranked by R²
all_results = list(BASELINES.values())
for r in [res_vanilla, res_gatv2]:
    if r is not None:
        all_results.append({k: r[k] for k in ['model','r2','mae','spearman','brier','ece']})

df_all = pd.DataFrame(all_results)
# de-duplicate on model name, keep first (BASELINES already unique)
df_all = df_all.drop_duplicates(subset='model', keep='first')
df_all = df_all[['model','r2','mae','spearman','brier','ece']].sort_values(
    'r2', ascending=False).reset_index(drop=True)

print('  FULL COMPARISON TABLE (test split, original BP scale, geographic split)')
print(df_all.to_string(index=False))

combined = TBL_DIR / 'phase5c_all_models_comparison.csv'
df_all.to_csv(combined, index=False)
print(f'\n  Saved: {combined.name}')

In [ ]:
# High-risk tail (Bin 5) for the two GAT baselines — the operationally critical bin
for r in [res_vanilla, res_gatv2]:
    if r is None:
        continue
    bins = binned_metrics(r['y_true_bp'], r['y_pred_bp'])
    b5 = [b for b in bins if b['bin'] == len(bins)]
    print(f"  {r['model']}:")
    if b5:
        b = b5[0]
        print(f"    Bin 5 [{b['bin_low']:.4f}, {b['bin_high']:.4f}]  "
              f"n={b['n']:,}  R²={b['r2']:+.3f}  MAE={b['mae']:.5f}  "
              f"Spearman={b['spearman']:.3f}")
    print()

In [ ]:
# Figure — ablation ladder R² and ECE side by side
if len(ladder_df) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    order = ladder_df.iloc[::-1]  # smallest at bottom
    labels = [v.split(':')[0].split('(')[0].strip() for v in order['variant']]

    ax = axes[0]
    ax.barh(labels, order['r2'], color='#377eb8')
    for i, v in enumerate(order['r2']):
        ax.text(v, i, f' {v:.4f}', va='center', fontsize=10)
    ax.set_xlabel('R²'); ax.set_title('Architecture ablation — R² (higher better)')
    ax.grid(axis='x', alpha=0.3)

    ax2 = axes[1]
    ax2.barh(labels, order['ece'], color='#e41a1c')
    for i, v in enumerate(order['ece']):
        ax2.text(v, i, f' {v:.4f}', va='center', fontsize=10)
    ax2.set_xlabel('ECE'); ax2.set_title('Architecture ablation — ECE (lower better)')
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    fig_out = FIG_DIR / 'p5c_ablation_ladder.png'
    plt.savefig(fig_out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Figure saved: {fig_out.name}')

In [ ]:
print('='*55)
print('  PHASE 5C COMPLETION CHECKLIST')
print('='*55)

items = [
    ('Vanilla GAT trained',   (CKPT_DIR / 'gnn_gat_vanilla_best.pt').exists()),
    ('GATv2 trained',         (CKPT_DIR / 'gnn_gatv2_best.pt').exists()),
    ('Vanilla GAT evaluated', res_vanilla is not None),
    ('GATv2 evaluated',       res_gatv2 is not None),
    ('Ablation ladder saved', (TBL_DIR / 'phase5c_ablation_ladder.csv').exists()),
    ('No geographic leakage', int((graph.train_mask & graph.test_mask).sum()) == 0),
    ('Full comparison saved', (TBL_DIR / 'phase5c_all_models_comparison.csv').exists()),
]
all_ok = True
for label, ok in items:
    print(f'  {"✓" if ok else "✗"}  {label}')
    all_ok = all_ok and ok
print('='*55)
print('  ALL CHECKS PASSED — ready for cross-validation (Phase 5E)'
      if all_ok else '  SOME CHECKS FAILED — see above')

# Phase 5C — GAT Baselines & Architecture Ablation · Documentation
## For Research Publication

*(Full paper-ready documentation follows below — fill the confirmed numbers
from the cells above once both models have finished training.)*